In [ ]:
USE ROLE ROLE_TEAM_QUERYQUEST;
USE DATABASE DB_TEAM_QUERYQUEST;
USE WAREHOUSE Animal_Task_WH;

-- Create the schema for raw data
CREATE SCHEMA IF NOT EXISTS BRONZE;

-- Create a stage to hold the CSV files before loading
CREATE STAGE IF NOT EXISTS project_stage;

USE SCHEMA BRONZE;



In [ ]:
-- Define File Format: Handles CSV specificities like headers and null values
CREATE OR REPLACE FILE FORMAT my_csv_format
    TYPE = 'CSV'
    SKIP_HEADER = 1 -- Skip the column name row
    FIELD_OPTIONALLY_ENCLOSED_BY = '"' -- Handle commas inside text fields
    NULL_IF = ('NULL', 'null', '') -- Standardize missing data
    EMPTY_FIELD_AS_NULL = TRUE;
    
-- Create the table and load the initial historical dataset in one step
CREATE OR REPLACE TABLE IMR_RAW AS
SELECT
    $1 AS Reference_ID,
    $2 AS Report_Year,
    $3 AS Diagnosis_Category,
    $4 AS Diagnosis_Sub_Category,
    $5 AS Treatment_Category,
    $6 AS Treatment_Sub_Category,
    $7 AS Determination,
    $8 AS Review_Type,
    $9 AS Age_Range,
    $10 AS Patient_Gender,
    $11 AS Findings
FROM @PROJECT_STAGE/Cal_Independent_Medical_Reviews.csv
(FILE_FORMAT => my_csv_format);

In [ ]:
-- Add columns to store AI outputs
ALTER TABLE IMR_RAW ADD COLUMN SENTIMENT FLOAT;
ALTER TABLE IMR_RAW ADD COLUMN SUMMARY STRING;

-- Compute AI insights (Sentiment & Summary) for all rows
UPDATE IMR_RAW
SET 
    SENTIMENT = SNOWFLAKE.CORTEX.SENTIMENT(Findings),
    SUMMARY = SNOWFLAKE.CORTEX.SUMMARIZE(Findings);

-- Verification
SELECT Reference_ID, Findings, SENTIMENT, SUMMARY 
FROM IMR_RAW 
LIMIT 10;

In [ ]:
-- Build an index on the text column to allow for natural language searching
CREATE OR REPLACE CORTEX SEARCH SERVICE IMR_SEARCH_SERVICE
ON Findings
ATTRIBUTES Diagnosis_Category, Report_Year
WAREHOUSE = Animal_Task_WH
TARGET_LAG = '1 minute' -- Data freshness target
AS (
    SELECT
        Findings,
        Diagnosis_Category,
        Report_Year
    FROM BRONZE.IMR_RAW
);


In [ ]:
-- Test Query 1: Concept search ("heart failure")
WITH search_results AS (
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'IMR_SEARCH_SERVICE',
            '{
               "query": "heart failure",
               "columns": [
                  "Findings",
                  "Diagnosis_Category",
                  "Report_Year"
               ],
               "limit": 10
            }'
        )
    ) AS json_data
)
SELECT
    r.value:"Diagnosis_Category"::STRING AS Diagnosis_Category,
    r.value:"Report_Year"::STRING AS Report_Year,
    r.value:"Findings"::STRING AS Findings,
    r.value:"@scores":"cosine_similarity"::FLOAT AS cosine_similarity,
    r.value:"@scores":"text_match"::FLOAT AS text_match
FROM search_results,
     LATERAL FLATTEN(input => json_data['results']) r;

In [ ]:
-- Test Query 2: Search with strict filtering (Year 2016)
WITH search_results AS (
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'IMR_SEARCH_SERVICE',
            '{
               "query": "surgery complications",
               "columns": [
                  "Findings",
                  "Diagnosis_Category",
                  "Report_Year"
               ],
               "filter": {"@eq": {"Report_Year": "2016"} },
               "limit": 10
            }'
        )
    ) AS json_data
)
SELECT
    r.value:"Diagnosis_Category"::STRING AS Diagnosis_Category,
    r.value:"Report_Year"::STRING AS Report_Year,
    r.value:"Findings"::STRING AS Findings,
    r.value:"@scores":"cosine_similarity"::FLOAT AS cosine_similarity,
    r.value:"@scores":"text_match"::FLOAT AS text_match
FROM search_results,
     LATERAL FLATTEN(input => json_data['results']) r;

In [ ]:
-- Test Query 3: Complex Search (Keywords + Year Filter)
-- Demonstrates searching for records containing BOTH "neurology" and "surgery"
-- while strictly filtering for the Report Year 2018.

USE SCHEMA BRONZE;
WITH search_results AS (
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'IMR_SEARCH_SERVICE',
            '{
               "query": "neurology AND surgery",
               "columns": [
                  "Findings",
                  "Diagnosis_Category",
                  "Report_Year"
               ],
               "filter": {
                   "@eq": { "Report_Year": "2016" }
               },
               "limit": 10
            }'
        )
    ) AS json_data
)
SELECT
    r.value:"Diagnosis_Category"::STRING AS Diagnosis_Category,
    r.value:"Report_Year"::INT AS Report_Year,
    r.value:"Findings"::STRING AS Findings,
    r.value:"@scores":"cosine_similarity"::FLOAT AS cosine_similarity,
    r.value:"@scores":"text_match"::FLOAT AS text_match
FROM search_results,
     LATERAL FLATTEN(input => json_data['results']) r
ORDER BY cosine_similarity DESC;


In [ ]:
USE ROLE ROLE_TEAM_QUERYQUEST;
USE DATABASE DB_TEAM_QUERYQUEST;
USE WAREHOUSE Animal_Task_WH;

-- Create Silver layer schema
CREATE SCHEMA IF NOT EXISTS SILVER;

USE SCHEMA SILVER;

In [ ]:
-- Create Sequences for the Dimension tables 
CREATE OR REPLACE SEQUENCE seq_date_sk START WITH 1 INCREMENT BY 1;
CREATE OR REPLACE SEQUENCE seq_diagnosis_sk START WITH 100 INCREMENT BY 5;
CREATE OR REPLACE SEQUENCE seq_treatment_sk START WITH 200 INCREMENT BY 7;
CREATE OR REPLACE SEQUENCE seq_review_sk START WITH 300 INCREMENT BY 2;
CREATE OR REPLACE SEQUENCE seq_patient_sk START WITH 400 INCREMENT BY 1;

In [ ]:
-- ============================================
-- DIMENSION TABLES
-- ============================================

-- 1. DIM_DIAGNOSIS
CREATE OR REPLACE TABLE DIM_DIAGNOSIS (
    Diagnosis_SK INT PRIMARY KEY,
    Diagnosis_Category VARCHAR(255) NOT NULL, 
    Diagnosis_Sub_Category VARCHAR(255) NOT NULL, 
    CONSTRAINT UK_DIAGNOSIS UNIQUE (Diagnosis_Category, Diagnosis_Sub_Category)
);

-- 2. DIM_TREATMENT
CREATE OR REPLACE TABLE DIM_TREATMENT (
    Treatment_SK INT PRIMARY KEY,
    Treatment_Category VARCHAR(255) NOT NULL, 
    Treatment_Sub_Category VARCHAR(255) NOT NULL, 
    CONSTRAINT UK_TREATMENT UNIQUE (Treatment_Category, Treatment_Sub_Category)
);

-- 3. DIM_PATIENT
CREATE OR REPLACE TABLE DIM_PATIENT (
    Patient_SK INT PRIMARY KEY,
    Age_Range VARCHAR(50) NOT NULL, 
    Patient_Gender VARCHAR(50) NOT NULL, 
    CONSTRAINT UK_PATIENT UNIQUE (Age_Range, Patient_Gender)
);

-- 4. DIM_REVIEW
CREATE OR REPLACE TABLE DIM_REVIEW (
    Review_SK INT PRIMARY KEY,
    Review_Type VARCHAR(255) NOT NULL, 
    Determination VARCHAR(255) NOT NULL, 
    CONSTRAINT UK_REVIEW UNIQUE (Review_Type, Determination)
);

-- 5. DIM_DATE
CREATE OR REPLACE TABLE DIM_DATE (
    Date_SK INT PRIMARY KEY,
    Report_Year INT NOT NULL,
    Decade INT NOT NULL,
    Four_Year_Bin VARCHAR(20) NOT NULL,
    Four_Year_Bin_Start INT NOT NULL,
    Four_Year_Bin_End INT NOT NULL,
    CONSTRAINT UK_DATE UNIQUE (Report_Year)
);

In [ ]:
-- ============================================
-- FACT TABLE
-- ============================================

CREATE OR REPLACE TABLE FACT_IMR (
    Reference_ID VARCHAR(50) PRIMARY KEY,
    Diagnosis_SK INT,
    Treatment_SK INT,
    Patient_SK INT,
    Review_SK INT,
    Date_SK INT,
    Findings TEXT,
    Sentiment FLOAT,
    Summary TEXT,
    -- Foreign Keys
    CONSTRAINT FK_DIAGNOSIS FOREIGN KEY (Diagnosis_SK) REFERENCES DIM_DIAGNOSIS(Diagnosis_SK),
    CONSTRAINT FK_TREATMENT FOREIGN KEY (Treatment_SK) REFERENCES DIM_TREATMENT(Treatment_SK),
    CONSTRAINT FK_PATIENT FOREIGN KEY (Patient_SK) REFERENCES DIM_PATIENT(Patient_SK),
    CONSTRAINT FK_REVIEW FOREIGN KEY (Review_SK) REFERENCES DIM_REVIEW(Review_SK),
    CONSTRAINT FK_DATE FOREIGN KEY (Date_SK) REFERENCES DIM_DATE(Date_SK)
);


In [ ]:
-- ============================================
-- POPULATE DIMENSION TABLES
-- ============================================

-- Populate DIM_DIAGNOSIS
INSERT INTO DIM_DIAGNOSIS (Diagnosis_SK, Diagnosis_Category, Diagnosis_Sub_Category)
SELECT 
    seq_diagnosis_sk.NEXTVAL AS Diagnosis_SK,
    COALESCE(t.Diagnosis_Category, 'Unspecified') AS Diagnosis_Category, 
    COALESCE(t.Diagnosis_Sub_Category, 'Unspecified') AS Diagnosis_Sub_Category
FROM (
    SELECT DISTINCT TRIM(Diagnosis_Category) AS Diagnosis_Category, TRIM(Diagnosis_Sub_Category) AS Diagnosis_Sub_Category
    FROM BRONZE.IMR_RAW
) t
ORDER BY Diagnosis_Category, Diagnosis_Sub_Category;


-- Populate DIM_TREATMENT 
INSERT INTO DIM_TREATMENT (Treatment_SK, Treatment_Category, Treatment_Sub_Category)
SELECT 
    seq_treatment_sk.NEXTVAL AS Treatment_SK,
    COALESCE(t.Treatment_Category, 'Unspecified') AS Treatment_Category, 
    COALESCE(t.Treatment_Sub_Category, 'Unspecified') AS Treatment_Sub_Category
FROM (
    SELECT DISTINCT TRIM(Treatment_Category) AS Treatment_Category, TRIM(Treatment_Sub_Category) AS Treatment_Sub_Category
    FROM BRONZE.IMR_RAW
) t
ORDER BY Treatment_Category, Treatment_Sub_Category;


-- Populate DIM_PATIENT 
INSERT INTO DIM_PATIENT (Patient_SK, Age_Range, Patient_Gender)
SELECT 
    seq_patient_sk.NEXTVAL AS Patient_SK,
    COALESCE(t.Age_Range, 'Unspecified') AS Age_Range,        
    COALESCE(t.Patient_Gender, 'Unspecified') AS Patient_Gender 
FROM (
    SELECT DISTINCT TRIM(Age_Range) AS Age_Range, TRIM(Patient_Gender) AS Patient_Gender
    FROM BRONZE.IMR_RAW
) t
ORDER BY Age_Range, Patient_Gender;


-- Populate DIM_REVIEW 
INSERT INTO DIM_REVIEW (Review_SK, Review_Type, Determination)
SELECT 
    seq_review_sk.NEXTVAL AS Review_SK,
    COALESCE(t.Review_Type, 'Unspecified') AS Review_Type,         
    COALESCE(t.Determination, 'Unspecified') AS Determination     
FROM (
    SELECT DISTINCT TRIM(Review_Type) AS Review_Type, TRIM(Determination) AS Determination
    FROM BRONZE.IMR_RAW
) t
ORDER BY Review_Type, Determination;

-- Populate DIM_DATE 
INSERT INTO DIM_DATE (
    Date_SK, 
    Report_Year, 
    Decade, Four_Year_Bin, Four_Year_Bin_Start, Four_Year_Bin_End
)
SELECT 
    seq_date_sk.NEXTVAL AS Date_SK,
    Report_Year, Decade, Four_Year_Bin, Four_Year_Bin_Start, Four_Year_Bin_End
FROM (
    SELECT DISTINCT 
        CAST(Report_Year AS INT) AS Report_Year,
        FLOOR(CAST(Report_Year AS INT) / 10) * 10 AS Decade,
        CONCAT(FLOOR((CAST(Report_Year AS INT) - 2001) / 4) * 4 + 2001, '-', FLOOR((CAST(Report_Year AS INT) - 2001) / 4) * 4 + 2004) AS Four_Year_Bin,
        FLOOR((CAST(Report_Year AS INT) - 2001) / 4) * 4 + 2001 AS Four_Year_Bin_Start,
        FLOOR((CAST(Report_Year AS INT) - 2001) / 4) * 4 + 2004 AS Four_Year_Bin_End
    FROM BRONZE.IMR_RAW
    WHERE Report_Year IS NOT NULL 
    ORDER BY Report_Year
);


In [ ]:

-- ============================================
-- POPULATE FACT TABLE
-- ============================================

INSERT INTO FACT_IMR (
    Reference_ID, Diagnosis_SK, Treatment_SK, Patient_SK, Review_SK, Date_SK, 
    Findings, Sentiment, Summary
)
SELECT 
    b.Reference_ID, dd.Diagnosis_SK, dt.Treatment_SK, dp.Patient_SK, dr.Review_SK, ddate.Date_SK, 
    b.Findings, b.Sentiment, b.Summary
FROM BRONZE.IMR_RAW b

-- DIM_DIAGNOSIS:
INNER JOIN DIM_DIAGNOSIS dd 
    ON COALESCE(TRIM(b.Diagnosis_Category), 'Unspecified') = dd.Diagnosis_Category 
    AND COALESCE(TRIM(b.Diagnosis_Sub_Category), 'Unspecified') = dd.Diagnosis_Sub_Category

-- DIM_TREATMENT: 
INNER JOIN DIM_TREATMENT dt 
    ON COALESCE(TRIM(b.Treatment_Category), 'Unspecified') = dt.Treatment_Category 
    AND COALESCE(TRIM(b.Treatment_Sub_Category), 'Unspecified') = dt.Treatment_Sub_Category

-- DIM_PATIENT:
INNER JOIN DIM_PATIENT dp 
    ON COALESCE(TRIM(b.Age_Range), 'Unspecified') = dp.Age_Range 
    AND COALESCE(TRIM(b.Patient_Gender), 'Unspecified') = dp.Patient_Gender

-- DIM_REVIEW: 
INNER JOIN DIM_REVIEW dr 
    ON COALESCE(TRIM(b.Review_Type), 'Unspecified') = dr.Review_Type 
    AND COALESCE(TRIM(b.Determination), 'Unspecified') = dr.Determination

-- DIM_DATE:
INNER JOIN DIM_DATE ddate 
    ON CAST(b.Report_Year AS INT) = ddate.Report_Year;


--Verification
SELECT * FROM DIM_DATE;
SELECT * FROM DIM_DIAGNOSIS;
SELECT * FROM DIM_PATIENT;
SELECT * FROM DIM_REVIEW;
SELECT * FROM DIM_TREATMENT;
    
SELECT * FROM FACT_IMR;

SELECT * FROM BRONZE.IMR_RAW;


In [ ]:
-- ============================================
-- DATA QUALITY CHECKS
-- ============================================

-- 1. Verify Reference_ID uniqueness in Bronze layer
SELECT 
    'Reference_ID Uniqueness Check' AS Check_Name,
    COUNT(*) AS Total_Records,
    COUNT(DISTINCT Reference_ID) AS Unique_Reference_IDs,
    COUNT(*) - COUNT(DISTINCT Reference_ID) AS Duplicates
FROM BRONZE.IMR_RAW;

In [ ]:
-- 2. Check row counts across layers
SELECT 'BRONZE.IMR_RAW' AS Table_Name, COUNT(*) AS Row_Count FROM BRONZE.IMR_RAW
UNION ALL
SELECT 'SILVER.FACT_IMR', COUNT(*) FROM SILVER.FACT_IMR
UNION ALL
SELECT 'SILVER.DIM_DIAGNOSIS', COUNT(*) FROM SILVER.DIM_DIAGNOSIS
UNION ALL
SELECT 'SILVER.DIM_TREATMENT', COUNT(*) FROM SILVER.DIM_TREATMENT
UNION ALL
SELECT 'SILVER.DIM_PATIENT', COUNT(*) FROM SILVER.DIM_PATIENT
UNION ALL
SELECT 'SILVER.DIM_REVIEW', COUNT(*) FROM SILVER.DIM_REVIEW
UNION ALL
SELECT 'SILVER.DIM_DATE', COUNT(*) FROM SILVER.DIM_DATE;

In [ ]:
-- 3. Check for NULL foreign keys
SELECT 
    'NULL Diagnosis_SK' AS Issue, COUNT(*) AS Count
FROM SILVER.FACT_IMR WHERE Diagnosis_SK IS NULL
UNION ALL
SELECT 'NULL Treatment_SK', COUNT(*) FROM SILVER.FACT_IMR WHERE Treatment_SK IS NULL
UNION ALL
SELECT 'NULL Patient_SK', COUNT(*) FROM SILVER.FACT_IMR WHERE Patient_SK IS NULL
UNION ALL
SELECT 'NULL Review_SK', COUNT(*) FROM SILVER.FACT_IMR WHERE Review_SK IS NULL
UNION ALL
SELECT 'NULL Date_SK', COUNT(*) FROM SILVER.FACT_IMR WHERE Date_SK IS NULL;

In [ ]:
--define the role, database, and warehouse
use role role_team_queryquest;
use database db_team_queryquest;
use warehouse animal_task_wh;

In [ ]:
--create gold layer
create schema if not exists gold;
use schema gold;


In [ ]:
--USE CASE 1 - IMR Outcomes Over Time
--Tracks yearly patterns in upheld vs overturned decisions, helping identify shifts in decision fairness or review consistency.
CREATE OR REPLACE TABLE GOLD.IMR_YEAR_DETERMINATION AS
SELECT
    d.Report_Year,
    r.Determination,
    COUNT(*) AS Num_Reviews,
    AVG(f.Sentiment) AS Avg_Sentiment
FROM SILVER.FACT_IMR f
JOIN SILVER.DIM_DATE d ON f.Date_SK = d.Date_SK
JOIN SILVER.DIM_REVIEW r ON f.Review_SK = r.Review_SK
GROUP BY d.Report_Year, r.Determination
ORDER BY d.Report_Year, r.Determination
;



In [ ]:
SELECT * FROM GOLD.IMR_YEAR_DETERMINATION LIMIT 10;

In [ ]:
SELECT COUNT(*) FROM GOLD.IMR_YEAR_DETERMINATION;  

In [ ]:
--USE CASE 2 — Clinical Patterns (Diagnosis × Treatment)
--Highlights diagnosis–treatment areas with higher overturn counts, revealing potential misalignment between policy criteria and clinical needs.
CREATE OR REPLACE TABLE GOLD.IMR_DIAG_TREAT_OUTCOME AS
SELECT
    ddate.Report_Year,
    dd.Diagnosis_Category,
    dt.Treatment_Category,
    r.Determination,
    COUNT(*) AS Num_Reviews,
    AVG(f.Sentiment) AS Avg_Sentiment
FROM SILVER.FACT_IMR f
JOIN SILVER.DIM_DATE ddate ON f.Date_SK = ddate.Date_SK
JOIN SILVER.DIM_DIAGNOSIS dd ON f.Diagnosis_SK = dd.Diagnosis_SK
JOIN SILVER.DIM_TREATMENT dt ON f.Treatment_SK = dt.Treatment_SK
JOIN SILVER.DIM_REVIEW r ON f.Review_SK = r.Review_SK
GROUP BY ddate.Report_Year, dd.Diagnosis_Category, dt.Treatment_Category, r.Determination
ORDER BY ddate.Report_Year, dd.Diagnosis_Category, dt.Treatment_Category, r.Determination
;

In [ ]:
SELECT * FROM GOLD.IMR_DIAG_TREAT_OUTCOME LIMIT 10;

In [ ]:
SELECT COUNT(*) FROM GOLD.IMR_DIAG_TREAT_OUTCOME; 

In [ ]:
--USE CASE 3 — Demographic Insights (Age × Gender)
--Examines outcome differences across age ranges and genders to detect potential disparities or patterns in IMR decisions.
CREATE OR REPLACE TABLE GOLD.IMR_DEMOGRAPHICS_OUTCOME AS
SELECT
    ddate.Report_Year,
    p.Age_Range,
    p.Patient_Gender,
    r.Determination,
    COUNT(*) AS Num_Reviews,
    AVG(f.Sentiment) AS Avg_Sentiment
FROM SILVER.FACT_IMR f
JOIN SILVER.DIM_DATE ddate ON f.Date_SK = ddate.Date_SK
JOIN SILVER.DIM_PATIENT p ON f.Patient_SK = p.Patient_SK
JOIN SILVER.DIM_REVIEW r ON f.Review_SK = r.Review_SK
GROUP BY ddate.Report_Year, p.Age_Range, p.Patient_Gender, r.Determination
ORDER BY ddate.Report_Year, p.Age_Range, p.Patient_Gender, r.Determination;

In [ ]:
SELECT * FROM GOLD.IMR_DEMOGRAPHICS_OUTCOME LIMIT 10;

In [ ]:
SELECT COUNT(*) FROM GOLD.IMR_DEMOGRAPHICS_OUTCOME;

In [ ]:
SELECT 'Before Load' as Status, count(*) as Count FROM SILVER.FACT_IMR;


In [ ]:
SELECT 'Before Load' as Status, count(*) as Count FROM GOLD.IMR_YEAR_DETERMINATION;

In [ ]:
SELECT 'Before Load' as Status, count(*) as Count FROM GOLD.IMR_DIAG_TREAT_OUTCOME;

In [ ]:
SELECT 'Before Load' as Status, count(*) as Count FROM GOLD.IMR_DEMOGRAPHICS_OUTCOME;

In [ ]:
USE SCHEMA BRONZE;

-- =========================================================
-- INGEST INCREMENTAL DATA
-- =========================================================

-- Load only the new file into the existing Bronze table.
-- We explicitly map columns and set AI columns to NULL initially.

COPY INTO IMR_RAW
FROM (
    SELECT
        $1 AS Reference_ID,
        $2 AS Report_Year,
        $3 AS Diagnosis_Category,
        $4 AS Diagnosis_Sub_Category,
        $5 AS Treatment_Category,
        $6 AS Treatment_Sub_Category,
        $7 AS Determination,
        $8 AS Review_Type,
        $9 AS Age_Range,
        $10 AS Patient_Gender,
        $11 AS Findings,
        NULL AS Sentiment, -- Placeholder for AI processing
        NULL AS Summary    -- Placeholder for AI processing
    FROM @PROJECT_STAGE/Incremental_Load.csv
    )
FILE_FORMAT = my_csv_format;

In [ ]:
-- The new rows have reference ids starting with SAM
Select * from IMR_RAW where reference_Id like '%SAM%';

In [ ]:
-- Only run Cortex functions on rows where Sentiment is NULL.
-- This prevents re-processing the entire historical dataset.

UPDATE IMR_RAW
SET 
    SENTIMENT = SNOWFLAKE.CORTEX.SENTIMENT(Findings),
    SUMMARY = SNOWFLAKE.CORTEX.SUMMARIZE(Findings)
WHERE SENTIMENT IS NULL;

In [ ]:
-- =========================================================
-- UPDATE SILVER DIMENSIONS 
-- =========================================================

USE SCHEMA SILVER;

-- Strategy for all Dimensions:
-- 1. Identify distinct values in Bronze.
-- 2. Check if they already exist in Silver (NOT IN clause).
-- 3. If new, generate a new Surrogate Key (NEXTVAL) and insert.

-- Update DIM_DIAGNOSIS

INSERT INTO DIM_DIAGNOSIS (Diagnosis_SK, Diagnosis_Category, Diagnosis_Sub_Category)
SELECT 
    seq_diagnosis_sk.NEXTVAL,
    New_Diag.Diagnosis_Category,
    New_Diag.Diagnosis_Sub_Category
FROM (
    SELECT DISTINCT 
        COALESCE(TRIM(Diagnosis_Category), 'Unspecified') AS Diagnosis_Category, 
        COALESCE(TRIM(Diagnosis_Sub_Category), 'Unspecified') AS Diagnosis_Sub_Category
    FROM BRONZE.IMR_RAW
    WHERE (Diagnosis_Category, Diagnosis_Sub_Category) NOT IN (
        SELECT Diagnosis_Category, Diagnosis_Sub_Category FROM DIM_DIAGNOSIS
    )
) New_Diag;

In [ ]:
-- Verification: Show the newly added Dimension keys

SELECT * FROM SILVER.DIM_DIAGNOSIS 
ORDER BY Diagnosis_SK DESC 
LIMIT 10;

In [ ]:
-- Update DIM_TREATMENT

INSERT INTO DIM_TREATMENT (Treatment_SK, Treatment_Category, Treatment_Sub_Category)
SELECT 
    seq_treatment_sk.NEXTVAL,
    New_Treat.Treatment_Category,
    New_Treat.Treatment_Sub_Category
FROM (
    SELECT DISTINCT 
        COALESCE(TRIM(Treatment_Category), 'Unspecified') AS Treatment_Category, 
        COALESCE(TRIM(Treatment_Sub_Category), 'Unspecified') AS Treatment_Sub_Category
    FROM BRONZE.IMR_RAW
    WHERE (Treatment_Category, Treatment_Sub_Category) NOT IN (
        SELECT Treatment_Category, Treatment_Sub_Category FROM DIM_TREATMENT
    )
) New_Treat;

In [ ]:
SELECT * FROM SILVER.DIM_TREATMENT
ORDER BY Treatment_SK DESC 
LIMIT 10;

In [ ]:
-- Update DIM_PATIENT
INSERT INTO DIM_PATIENT (Patient_SK, Age_Range, Patient_Gender)
SELECT 
    seq_patient_sk.NEXTVAL,
    New_Pat.Age_Range,
    New_Pat.Patient_Gender
FROM (
    SELECT DISTINCT 
        COALESCE(TRIM(Age_Range), 'Unspecified') AS Age_Range, 
        COALESCE(TRIM(Patient_Gender), 'Unspecified') AS Patient_Gender
    FROM BRONZE.IMR_RAW
    WHERE (Age_Range, Patient_Gender) NOT IN (
        SELECT Age_Range, Patient_Gender FROM DIM_PATIENT
    )
) New_Pat;

In [ ]:
SELECT * FROM SILVER.DIM_PATIENT
ORDER BY Patient_SK DESC 
LIMIT 1;

In [ ]:
-- Update DIM_REVIEW
INSERT INTO DIM_REVIEW (Review_SK, Review_Type, Determination)
SELECT 
    seq_review_sk.NEXTVAL,
    New_Rev.Review_Type,
    New_Rev.Determination
FROM (
    SELECT DISTINCT 
        COALESCE(TRIM(Review_Type), 'Unspecified') AS Review_Type, 
        COALESCE(TRIM(Determination), 'Unspecified') AS Determination
    FROM BRONZE.IMR_RAW
    WHERE (Review_Type, Determination) NOT IN (
        SELECT Review_Type, Determination FROM DIM_REVIEW
    )
) New_Rev;

In [ ]:
SELECT * FROM SILVER.DIM_REVIEW
ORDER BY Review_SK DESC 
LIMIT 4;

In [ ]:
-- Update DIM_DATE (Handling Binning Logic)
INSERT INTO DIM_DATE (Date_SK, Report_Year, Decade, Four_Year_Bin, Four_Year_Bin_Start, Four_Year_Bin_End)
SELECT 
    seq_date_sk.NEXTVAL,
    New_Date.Report_Year,
    FLOOR(New_Date.Report_Year / 10) * 10,
    CONCAT(FLOOR((New_Date.Report_Year - 2001) / 4) * 4 + 2001, '-', FLOOR((New_Date.Report_Year - 2001) / 4) * 4 + 2004),
    FLOOR((New_Date.Report_Year - 2001) / 4) * 4 + 2001,
    FLOOR((New_Date.Report_Year - 2001) / 4) * 4 + 2004
FROM (
    SELECT DISTINCT CAST(Report_Year AS INT) AS Report_Year
    FROM BRONZE.IMR_RAW
    WHERE Report_Year IS NOT NULL 
    AND CAST(Report_Year AS INT) NOT IN (SELECT Report_Year FROM DIM_DATE)
) New_Date;

In [ ]:
SELECT * FROM SILVER.DIM_DATE
ORDER BY Date_SK DESC 
LIMIT 6;

In [ ]:
-- =========================================================
-- UPDATE FACT TABLE (INCREMENTAL)
-- =========================================================
-- Insert only rows where the Reference_ID does not yet exist in the Fact table.
-- We join back to Dimensions to retrieve the correct SKs (old or new).

INSERT INTO FACT_IMR (
    Reference_ID, Diagnosis_SK, Treatment_SK, Patient_SK, Review_SK, Date_SK, 
    Findings, Sentiment, Summary
)
SELECT 
    b.Reference_ID, dd.Diagnosis_SK, dt.Treatment_SK, dp.Patient_SK, dr.Review_SK, ddate.Date_SK, 
    b.Findings, b.Sentiment, b.Summary
FROM BRONZE.IMR_RAW b
-- Join to standard dimensions to get SKs
INNER JOIN DIM_DIAGNOSIS dd 
    ON COALESCE(TRIM(b.Diagnosis_Category), 'Unspecified') = dd.Diagnosis_Category 
    AND COALESCE(TRIM(b.Diagnosis_Sub_Category), 'Unspecified') = dd.Diagnosis_Sub_Category
INNER JOIN DIM_TREATMENT dt 
    ON COALESCE(TRIM(b.Treatment_Category), 'Unspecified') = dt.Treatment_Category 
    AND COALESCE(TRIM(b.Treatment_Sub_Category), 'Unspecified') = dt.Treatment_Sub_Category
INNER JOIN DIM_PATIENT dp 
    ON COALESCE(TRIM(b.Age_Range), 'Unspecified') = dp.Age_Range 
    AND COALESCE(TRIM(b.Patient_Gender), 'Unspecified') = dp.Patient_Gender
INNER JOIN DIM_REVIEW dr 
    ON COALESCE(TRIM(b.Review_Type), 'Unspecified') = dr.Review_Type 
    AND COALESCE(TRIM(b.Determination), 'Unspecified') = dr.Determination
INNER JOIN DIM_DATE ddate 
    ON CAST(b.Report_Year AS INT) = ddate.Report_Year
-- Only insert rows that don't already exist in the Fact table
WHERE b.Reference_ID NOT IN (SELECT Reference_ID FROM FACT_IMR);

In [ ]:
SELECT * FROM SILVER.FACT_IMR
where Reference_ID like '%SAM%';

In [ ]:
-- Verification: Check totals after load
SELECT 'After Load' AS Status, 'FACT_IMR' AS Table_Name, COUNT(*) AS Row_Count FROM SILVER.FACT_IMR
UNION ALL
SELECT 'After Load', 'DIM_DIAGNOSIS', COUNT(*) FROM SILVER.DIM_DIAGNOSIS;

In [ ]:
-- =========================================================
-- REFRESH GOLD LAYER 
-- =========================================================
-- Gold tables are aggregates, so we perform a full refresh to reflect new data.

USE SCHEMA GOLD;

In [ ]:
USE ROLE role_team_queryquest;
USE DATABASE db_team_queryquest;
USE WAREHOUSE animal_task_wh;
USE SCHEMA GOLD;

-- Check gold layer status (before load)
SELECT 
    'Before Load' AS Status,
    'IMR_YEAR_DETERMINATION' AS Table_Name, 
    COUNT(*) AS Aggregated_Rows,
    SUM(Num_Reviews) AS Total_Reviews_Processed
FROM GOLD.IMR_YEAR_DETERMINATION

UNION ALL

SELECT 
    'Before Load',
    'IMR_DIAG_TREAT_OUTCOME', 
    COUNT(*) AS Aggregated_Rows,
    SUM(Num_Reviews) AS Total_Reviews_Processed
FROM GOLD.IMR_DIAG_TREAT_OUTCOME

UNION ALL

SELECT 
    'Before Load',
    'IMR_DEMOGRAPHICS_OUTCOME', 
    COUNT(*) AS Aggregated_Rows,
    SUM(Num_Reviews) AS Total_Reviews_Processed
FROM GOLD.IMR_DEMOGRAPHICS_OUTCOME;

In [ ]:
-- Rebuild Gold Tables
CREATE OR REPLACE TABLE GOLD.IMR_YEAR_DETERMINATION AS
SELECT
    d.Report_Year,
    r.Determination,
    COUNT(*) AS Num_Reviews,
    AVG(f.Sentiment) AS Avg_Sentiment
FROM SILVER.FACT_IMR f
JOIN SILVER.DIM_DATE d ON f.Date_SK = d.Date_SK
JOIN SILVER.DIM_REVIEW r ON f.Review_SK = r.Review_SK
GROUP BY d.Report_Year, r.Determination
ORDER BY d.Report_Year, r.Determination;

In [ ]:
CREATE OR REPLACE TABLE GOLD.IMR_DIAG_TREAT_OUTCOME AS
SELECT
    ddate.Report_Year,
    dd.Diagnosis_Category,
    dt.Treatment_Category,
    r.Determination,
    COUNT(*) AS Num_Reviews,
    AVG(f.Sentiment) AS Avg_Sentiment
FROM SILVER.FACT_IMR f
JOIN SILVER.DIM_DATE ddate ON f.Date_SK = ddate.Date_SK
JOIN SILVER.DIM_DIAGNOSIS dd ON f.Diagnosis_SK = dd.Diagnosis_SK
JOIN SILVER.DIM_TREATMENT dt ON f.Treatment_SK = dt.Treatment_SK
JOIN SILVER.DIM_REVIEW r ON f.Review_SK = r.Review_SK
GROUP BY ddate.Report_Year, dd.Diagnosis_Category, dt.Treatment_Category, r.Determination
ORDER BY ddate.Report_Year, dd.Diagnosis_Category, dt.Treatment_Category, r.Determination;

In [ ]:
CREATE OR REPLACE TABLE GOLD.IMR_DEMOGRAPHICS_OUTCOME AS
SELECT
    ddate.Report_Year,
    p.Age_Range,
    p.Patient_Gender,
    r.Determination,
    COUNT(*) AS Num_Reviews,
    AVG(f.Sentiment) AS Avg_Sentiment
FROM SILVER.FACT_IMR f
JOIN SILVER.DIM_DATE ddate ON f.Date_SK = ddate.Date_SK
JOIN SILVER.DIM_PATIENT p ON f.Patient_SK = p.Patient_SK
JOIN SILVER.DIM_REVIEW r ON f.Review_SK = r.Review_SK
GROUP BY ddate.Report_Year, p.Age_Range, p.Patient_Gender, r.Determination
ORDER BY ddate.Report_Year, p.Age_Range, p.Patient_Gender, r.Determination;

In [ ]:
-- Final Checks: Verify gold layer status (after load)

SELECT 
    'IMR_YEAR_DETERMINATION' AS Table_Name, 
    SUM(Num_Reviews) AS Total_Reviews_Tracked 
FROM GOLD.IMR_YEAR_DETERMINATION
UNION ALL
SELECT 
    'IMR_DIAG_TREAT_OUTCOME', 
    SUM(Num_Reviews) 
FROM GOLD.IMR_DIAG_TREAT_OUTCOME
UNION ALL
SELECT 
    'IMR_DEMOGRAPHICS_OUTCOME', 
    SUM(Num_Reviews) 
FROM GOLD.IMR_DEMOGRAPHICS_OUTCOME;